# 📘 CIFAR-10 Image Classification Learning Project
## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

#  Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

#  Load Dataset
We use **CIFAR-10**, which contains **60,000 color images of size 32×32×3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

##  Visualize Sample Images

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

#  Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

For the ANN we also flatten each 32×32×3 image into a 3072-length vector, since dense layers expect 1D input.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm  = x_test  / 255.0

# Flat versions for ANN
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat  = x_test_norm.reshape(len(x_test_norm),  -1)

print("Normalised train shape (CNN):", x_train_norm.shape)
print("Flattened train shape  (ANN):", x_train_flat.shape)

# 🔹 Part 1: ANN Model

An ANN treats an image as a plain list of numbers — it has no idea that
neighbouring pixels are related. That's the core limitation. The model
below has **three hidden layers** (512 → 256 → 128) with Dropout
between each to slow overfitting.

**Why this still won't be great:**  
Flattening throws away spatial structure. A cat's ear three pixels to
the left looks like a completely different input to a dense layer.

In [ ]:
# --- Beginner Task 1: added an extra hidden layer (128 units) compared to the starter ---
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='ANN')

ann_model.summary()

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --- Beginner Task 4: EarlyStopping so training stops before overfitting gets bad ---
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# --- Beginner Task 3: training for 20 epochs (ES will cut it short if needed) ---
ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"ANN  — Test Loss: {ann_test_loss:.4f}  |  Test Accuracy: {ann_test_acc:.4f}")

# 🔹 Part 2: CNN Model

A CNN keeps the image as a 2D grid and slides small filters (kernels)
across it. Each filter learns to detect one pattern — an edge, a curve,
a texture. Stacking convolution layers builds up increasingly complex
feature detectors.

**Architecture used here:**
- Three Conv2D blocks with increasing filter counts **(32 → 64 → 128)**
- BatchNormalization after each block to stabilise activations
- MaxPooling to reduce spatial size and add translation tolerance
- Dropout before the final Dense layer

This is why CNN performs much better for image tasks.

In [ ]:
# --- Beginner Task 2: filter sizes scaled 32 → 64 → 128 as suggested ---
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN')

cnn_model.summary()

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_cnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_cnn],
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN  — Test Loss: {cnn_test_loss:.4f}  |  Test Accuracy: {cnn_test_acc:.4f}")

##  Compare Learning Curves (ANN vs CNN)

Plotting both validation accuracy curves together makes it easy to see
how much faster the CNN learns and how much higher it peaks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Validation accuracy
axes[0].plot(ann_history.history['val_accuracy'], label='ANN Val Acc',  color='royalblue',  linewidth=2)
axes[0].plot(cnn_history.history['val_accuracy'], label='CNN Val Acc',  color='darkorange', linewidth=2)
axes[0].plot(ann_history.history['accuracy'],     label='ANN Train Acc', color='royalblue',  linewidth=1.2, linestyle='--', alpha=0.6)
axes[0].plot(cnn_history.history['accuracy'],     label='CNN Train Acc', color='darkorange', linewidth=1.2, linestyle='--', alpha=0.6)
axes[0].set_title('ANN vs CNN — Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation loss
axes[1].plot(ann_history.history['val_loss'], label='ANN Val Loss',  color='royalblue',  linewidth=2)
axes[1].plot(cnn_history.history['val_loss'], label='CNN Val Loss',  color='darkorange', linewidth=2)
axes[1].set_title('ANN vs CNN — Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Part 3: Augmented CNN (Beginner Task 5)

**Data augmentation** creates modified copies of training images on the
fly — flipped, rotated, or zoomed slightly. This forces the model to
learn features that are robust to small changes, which usually cuts
the gap between training accuracy and validation accuracy.

The augmentation layers only run during training; at inference time
they are skipped automatically.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name='augmentation')

aug_cnn_model = models.Sequential([
    data_augmentation,

    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='Augmented_CNN')

aug_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_aug = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_aug],
    verbose=1
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"Augmented CNN — Test Loss: {aug_test_loss:.4f}  |  Test Accuracy: {aug_test_acc:.4f}")

##  Augmented CNN vs Plain CNN — Learning Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cnn_history.history['val_accuracy'],     label='CNN Val Acc',          color='darkorange', linewidth=2)
plt.plot(aug_history.history['val_accuracy'],     label='Augmented CNN Val Acc', color='seagreen',   linewidth=2)
plt.plot(cnn_history.history['accuracy'],         label='CNN Train Acc',         color='darkorange', linewidth=1.2, linestyle='--', alpha=0.5)
plt.plot(aug_history.history['accuracy'],         label='Augmented Train Acc',   color='seagreen',   linewidth=1.2, linestyle='--', alpha=0.5)
plt.title('CNN vs Augmented CNN — Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#  Final Comparison Table

A side-by-side view of test accuracy for all three model variants.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN (3 layers, 20 epochs)",
              "CNN (32-64-128 filters, 20 epochs)",
              "Augmented CNN (20 epochs)"],
    "Test Accuracy": [round(ann_test_acc, 4),
                      round(cnn_test_acc, 4),
                      round(aug_test_acc, 4)],
    "Test Loss":     [round(ann_test_loss, 4),
                      round(cnn_test_loss, 4),
                      round(aug_test_loss, 4)]
})

display(comparison.style.highlight_max(subset=['Test Accuracy'], color='lightgreen')
                        .highlight_min(subset=['Test Loss'],     color='lightblue'))

##  Bar Chart — Test Accuracy Comparison

In [ ]:
models_list  = ['ANN', 'CNN', 'Augmented CNN']
accuracies   = [ann_test_acc, cnn_test_acc, aug_test_acc]
colors       = ['royalblue', 'darkorange', 'seagreen']

plt.figure(figsize=(8, 5))
bars = plt.bar(models_list, accuracies, color=colors, edgecolor='black', width=0.5)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.005,
             f'{acc:.3f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylim(0, 1.0)
plt.ylabel('Test Accuracy')
plt.title('Model Comparison — CIFAR-10 Test Accuracy')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Beginner Tasks — Summary

All five tasks from the assignment have been completed in this notebook:

| # | Task | Where it was done |
|---|------|-------------------|
| 1 | Increase ANN layers | Added a 128-unit hidden layer → ANN now has 512 → 256 → 128 → 10 |
| 2 | Scale CNN filters 32 → 64 → 128 | Done in the CNN model cell; also added `padding='same'` |
| 3 | Train for 20 epochs | `epochs=20` set on all three models; EarlyStopping decides the actual cut-off |
| 4 | Add EarlyStopping | `EarlyStopping(monitor='val_loss', patience=3)` added to ANN and CNN |
| 5 | Run augmented CNN training | Full `aug_cnn_model.fit(...)` executed with RandomFlip / RandomRotation / RandomZoom |

The comparison table and bar chart at the end show final test accuracy for all three variants.

# ✅ Conclusion

A few key takeaways from running this experiment:

- **ANN** gets to roughly 50–55% accuracy. Not bad for a fully-connected
  network, but it's hitting a ceiling because flattening destroys spatial
  relationships — the model can't tell that two pixels next to each other
  are related.

- **CNN** jumps significantly higher (~70–75%) using the same number of
  epochs. The convolutional filters learn local patterns (edges, colours,
  textures) and then the pooling layers build up a hierarchy of features.

- **Augmented CNN** tends to match or slightly beat the plain CNN on the
  test set, and the gap between train and validation accuracy is smaller,
  which means it's generalising better — the model has learned features
  that work regardless of small variations in the input.

- **EarlyStopping** is worth using by default. It saved compute time and
  prevented the models from memorising noise in the training data.

The biggest lesson: for image data, **always start with a CNN**. The
inductive bias it brings (local connectivity + weight sharing) is exactly
what image structure demands.